## Conversational AI -Chatbot!

## OpenAI_APi_Key 세팅

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


# OpenAi & Graio ChatInterface

In [3]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [4]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "당신은 친절한 AI 어시스턴트입니다. 반드시 한국어로만 답변하세요."

 And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


OK! Let's write a slightly better chat callback!

In [6]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [7]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


# one shot prompting 1

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [8]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [9]:
system_message = "당신은 옷 가게의 친절한 쇼핑 도우미입니다. 고객이 세일 중인 상품을 구매하도록 부드럽게 유도해야 합니다. \
모자는 60% 할인 중이며, 대부분의 다른 상품은 50% 할인 중입니다. \
예를 들어, 고객이 '모자를 사고 싶어요'라고 말하면, \
'좋아요! 저희 매장에 모자가 다양하게 있는데, 그 중 일부는 세일 행사 상품이에요.'와 같이 답변할 수 있습니다. \
고객이 무엇을 살지 모를 때는 모자 구매를 추천해 주세요."

In [11]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


# one shot prompting 2

In [13]:
system_message += "\n만약 고객이 신발을 찾는다면, 오늘은 신발 할인이 없다고 안내하고, \
모자 세일을 다시 한번 추천해 주세요!"

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


# one shot prompting 3

In [15]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    relevant_system_message = system_message
    if '벨트' in message.lower():
        relevant_system_message += " 이 매장은 벨트를 판매하지 않습니다. 벨트를 찾는 고객에게는 현재 세일 중인 다른 상품을 안내해 주세요."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [16]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.
